# 03 - BERT Model Comparison

Detaillierter Vergleich der drei BERT-Modelle (mBERT, GBERT, HateBERT).

**Inhalte:**
- Cross-Validation Ergebnisse analysieren
- Per-Class Performance vergleichen
- Statistische Signifikanztests
- Learning Curves aus Experiment 3
- Preprocessing Ablation aus Experiment 4

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Pfade hinzufügen
project_root = Path('..')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import METRICS_DIR, PLOTS_DIR, MODELS

print("Bibliotheken geladen.")

Bibliotheken geladen.


## 1. CV-Ergebnisse laden

In [2]:
# Lade alle Ergebnisse
results = {}
for model_name in MODELS.keys():
    path = METRICS_DIR / f"{model_name}_cv_results.json"
    if path.exists():
        with open(path, "r") as f:
            data = json.load(f)
            results[model_name] = data
            
# Überprüfe geladene Daten
print(f"Geladene Modelle: {list(results.keys())}")

for model, data in results.items():
    print(f"\nModel: {model}")
    print(f"  Accuracy: {data['accuracy_mean']:.4f} (+/- {data['accuracy_std']:.4f})")
    print(f"  F1-Macro: {data['f1_macro_mean']:.4f} (+/- {data['f1_macro_std']:.4f})")

Geladene Modelle: ['mBERT', 'GBERT', 'HateBERT']

Model: mBERT
  Accuracy: 0.7965 (+/- 0.0115)
  F1-Macro: 0.7684 (+/- 0.0173)

Model: GBERT
  Accuracy: 0.8331 (+/- 0.0194)
  F1-Macro: 0.8080 (+/- 0.0200)

Model: HateBERT
  Accuracy: 0.7123 (+/- 0.0114)
  F1-Macro: 0.6724 (+/- 0.0065)


## 2. Statistische Signifikanztests

In [3]:
# DataFrame für Vergleich erstellen
comparison_data = []

for model, data in results.items():
    row = {
        'Model': model,
        'Accuracy': data.get('accuracy_mean', 0),
        'Accuracy_Std': data.get('accuracy_std', 0),
        'F1_Macro': data.get('f1_macro_mean', 0),
        'F1_Macro_Std': data.get('f1_macro_std', 0),
        'Precision_Macro': data.get('precision_macro_mean', 0),
        'Recall_Macro': data.get('recall_macro_mean', 0),
        'Train_Time(s)': data.get('total_train_time_sec', 0)
    }
    comparison_data.append(row)

# Baseline-Daten hinzufügen (falls vorhanden)
baseline_file = METRICS_DIR / 'baseline_results.json'
if baseline_file.exists():
    with open(baseline_file, 'r') as f:
        baseline_data = json.load(f)
    if 'baselines' in baseline_data:
        for bl in baseline_data['baselines']:
            row = {
                'Model': bl['model'],
                'Accuracy': bl.get('accuracy', 0),
                'Accuracy_Std': 0, # Wurde evtl. nicht berechnet
                'F1_Macro': bl.get('f1_macro', 0),
                'F1_Macro_Std': 0,
                'Precision_Macro': 0,
                'Recall_Macro': 0,
                'Train_Time(s)': bl.get('time_sec', 0)
            }
            comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)

# Sortieren nach F1_Macro
df_comparison = df_comparison.sort_values(by='F1_Macro', ascending=False)

print("\n=== Modell-Ranking (F1 Macro) ===")
print(df_comparison[['Model', 'F1_Macro', 'Accuracy', 'Train_Time(s)']].to_string(index=False))


=== Modell-Ranking (F1 Macro) ===
              Model  F1_Macro  Accuracy  Train_Time(s)
              GBERT  0.808000  0.833140       0.000000
              mBERT  0.768437  0.796490    1508.463506
           HateBERT  0.672351  0.712272    1168.789040
         SVM+TF-IDF  0.623466  0.696251       0.000000
            Lexikon  0.536628  0.693907       0.000000
      LogReg+TF-IDF  0.528453  0.684827       0.000000
RandomForest+TF-IDF  0.450684  0.672818       0.000000
           Majority  0.396607  0.657293       0.000000


## 3. Per-Class Performance

In [4]:
# Preprocessing Ablation Ergebnisse laden (wenn vorhanden)
preprocessing_file = METRICS_DIR / 'preprocessing_ablation_results.json'

if preprocessing_file.exists():
    with open(preprocessing_file, 'r') as f:
        preprocessing_data = json.load(f)
    print(f"Preprocessing Ablation Daten geladen.")
    
    # Ergebnisse aus 'results' extrahieren
    results_dict = preprocessing_data.get('results', {})
    
    # Alle Varianten ausgeben
    print('\n=== Preprocessing Stats (Top 5) ===')
    sorted_res = sorted(results_dict.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True)
    for k, v in sorted_res[:5]:
        print(f"{k}: F1={v['f1_macro_mean']:.4f}")
else:
    print('Experiment 4 Daten nicht gefunden.')

Preprocessing Ablation Daten geladen.

=== Preprocessing Stats (Top 5) ===
full_preprocessing: F1=0.8097
remove_urls: F1=0.8053
normalize_usernames: F1=0.8050
original: F1=0.8036
remove_emojis: F1=0.8031


## 4. Learning Curve (Experiment 3)

In [5]:
# 4. Learning Curve (Experiment 3)
# Visualisierung ausgelagert in scripts/generate_all_plots.py
# Siehe: results/plots/learning_curve.png

print('\nPlot "learning_curve.png" wird durch scripts/generate_all_plots.py erstellt.')

# Deskriptiv:
data_size_path = METRICS_DIR / 'data_size_variation_results.json'

if data_size_path.exists():
    with open(data_size_path, 'r') as f:
        data_size_results = json.load(f)
    print('\n=== Data Efficiency Stats ===')
    res_dict = data_size_results.get("results", data_size_results)
    for size_key in sorted(res_dict.keys(), key=lambda x: float(x.strip('%'))):
        data = res_dict[size_key]
        print(f'{size_key} Daten: F1={data["f1_macro_mean"]:.4f}, Acc={data["accuracy_mean"]:.4f}')
else:
    print('Experiment 3 Daten nicht gefunden.')


Plot "learning_curve.png" wird durch scripts/generate_all_plots.py erstellt.

=== Data Efficiency Stats ===
10% Daten: F1=0.5214, Acc=0.5845
25% Daten: F1=0.7097, Acc=0.7651
50% Daten: F1=0.7880, Acc=0.8144
75% Daten: F1=0.7914, Acc=0.8208
100% Daten: F1=0.8097, Acc=0.8334


## 5. Preprocessing Ablation (Experiment 4)

In [6]:
# 5. Preprocessing Ablation (Experiment 4)
# Visualisierung ausgelagert in scripts/generate_all_plots.py
# Siehe: results/plots/preprocessing_ablation.png

print('\nPlot "preprocessing_ablation.png" wird durch scripts/generate_all_plots.py erstellt.')

# Deskriptiv:
preprocessing_path = METRICS_DIR / 'preprocessing_ablation_results.json'

if preprocessing_path.exists():
    with open(preprocessing_path, 'r') as f:
        preprocessing_data = json.load(f)
    print('\n=== Preprocessing Stats (Top 5) ===')
    results_dict = preprocessing_data.get('results', {})
    sorted_res = sorted(results_dict.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True)
    for k, v in sorted_res[:5]:
        print(f"{k}: F1={v['f1_macro_mean']:.4f}")
else:
    print('Experiment 4 Daten nicht gefunden.')


Plot "preprocessing_ablation.png" wird durch scripts/generate_all_plots.py erstellt.

=== Preprocessing Stats (Top 5) ===
full_preprocessing: F1=0.8097
remove_urls: F1=0.8053
normalize_usernames: F1=0.8050
original: F1=0.8036
remove_emojis: F1=0.8031


## 6. Zusammenfassung für Paper

In [7]:
print('\n' + '='*70)
print('ZUSAMMENFASSUNG FÜR HAUSARBEIT')
print('='*70)

# Bestes Modell
best_model_name = max(results.items(), key=lambda x: x[1]['f1_macro_mean'])[0]
best_model_data = results[best_model_name]

print(f'\nBestes Modell: {best_model_name}')
print(f'   F1 (Macro):  {best_model_data["f1_macro_mean"]:.4f} +/- {best_model_data["f1_macro_std"]:.4f}')
print(f'   Accuracy:    {best_model_data["accuracy_mean"]:.4f} +/- {best_model_data["accuracy_std"]:.4f}')

# Modell-Ranking
print('\nModell-Ranking (nach F1):')
sorted_models = sorted(results.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True)
for i, (model, data) in enumerate(sorted_models, 1):
    print(f'   {i}. {model}: {data["f1_macro_mean"]:.4f}')

# Key Findings
print('\nKey Findings:')
if 'GBERT' in results and 'mBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    mbert_f1 = results['mBERT']['f1_macro_mean']
    diff = gbert_f1 - mbert_f1
    print(f'   * GBERT übertrifft mBERT um {diff:.4f} F1-Punkte ({diff/mbert_f1*100:.1f}%)')

if 'GBERT' in results and 'HateBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    hatebert_f1 = results['HateBERT']['f1_macro_mean']
    diff = gbert_f1 - hatebert_f1
    print(f'   * GBERT übertrifft HateBERT um {diff:.4f} F1-Punkte ({diff/hatebert_f1*100:.1f}%)')

print(f'   * Language-specific model (GBERT) schlägt multilingual (mBERT)')
print(f'   * Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT')


ZUSAMMENFASSUNG FÜR HAUSARBEIT

Bestes Modell: GBERT
   F1 (Macro):  0.8080 +/- 0.0200
   Accuracy:    0.8331 +/- 0.0194

Modell-Ranking (nach F1):
   1. GBERT: 0.8080
   2. mBERT: 0.7684
   3. HateBERT: 0.6724

Key Findings:
   * GBERT übertrifft mBERT um 0.0396 F1-Punkte (5.1%)
   * GBERT übertrifft HateBERT um 0.1356 F1-Punkte (20.2%)
   * Language-specific model (GBERT) schlägt multilingual (mBERT)
   * Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT
